# DW 2110 vs Gradient vs Lat-1 Filter Simulation

Compare three debiased Whittle filters on direct target-grid circulant-embedding simulations:

- `DW_2110`: scalar spatial filter `2 -1 -1 0` from `debiased_whittle_2110`
- `DW_Gradient`: vector filter `(D_lat, D_lon)` from `debiased_whittle_grad_filter`
- `DW_Lat1`: latitude-only first-difference filter from `debiased_whittle_lat1`

The gradient model treats the data as a `2 * T` dimensional multivariate process and uses the matching covariance of differences, including cross-covariances between `D_lat` and `D_lon`. This simulation starts directly from the regular target grid, so it does not include high-resolution remapping or real-location missingness.


In [15]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import Parameter

GEMS_TCO_PATH = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
if GEMS_TCO_PATH not in sys.path:
    sys.path.append(GEMS_TCO_PATH)

from GEMS_TCO import configuration as config
from GEMS_TCO import debiased_whittle_2110 as dw2110
from GEMS_TCO import debiased_whittle_grad_filter as dwgrad
from GEMS_TCO import debiased_whittle_lat1 as dwlat1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64
DELTA_LAT = 0.044
DELTA_LON = 0.063
T_STEPS = 8
LAT_COL, LON_COL, VAL_COL, TIME_COL = 0, 1, 2, 3

print(f"Device: {DEVICE}")
print("Models: DW_2110, DW_Gradient, DW_Lat1")


Device: cpu
Models: DW_2110, DW_Gradient, DW_Lat1


## Simulation Settings

For quick local testing, shrink `lat_range` / `lon_range` or set `num_iters = 1`. The simulation is generated directly on the target regular grid; no high-resolution remapping or forced-grid reconstruction is used.


In [ ]:
true_dict = {
    "sigmasq": 10.0,
    "range_lat": 0.3,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.08,
    "advec_lon": -0.126,
    "nugget": 2.5,
}

# Full ozone target domain. The filtered grid should be about 113 x 158.
lat_range = [-3.0, 2.0]
lon_range = [121.0, 131.0]

num_iters = 30
init_noise = 0.7
lbfgs_steps = 8
lbfgs_eval = 15
lbfgs_history = 50
seed = 42

result_tag = "full_domain_advec_lon_neg0126_n30_3filters"

rng = np.random.default_rng(seed)
torch.manual_seed(seed)


In [17]:
def true_to_log_params(d):
    phi2 = 1.0 / d["range_lon"]
    phi1 = d["sigmasq"] * phi2
    phi3 = (d["range_lon"] / d["range_lat"]) ** 2
    phi4 = (d["range_lon"] / d["range_time"]) ** 2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        d["advec_lat"], d["advec_lon"], np.log(d["nugget"]),
    ]


def backmap_params(params):
    vals = [x.detach().cpu().item() if isinstance(x, torch.Tensor) else float(x) for x in params[:7]]
    phi2 = np.exp(vals[1])
    phi3 = np.exp(vals[2])
    phi4 = np.exp(vals[3])
    range_lon = 1.0 / phi2
    return {
        "sigmasq": np.exp(vals[0]) / phi2,
        "range_lat": range_lon / np.sqrt(phi3),
        "range_lon": range_lon,
        "range_time": range_lon / np.sqrt(phi4),
        "advec_lat": vals[4],
        "advec_lon": vals[5],
        "nugget": np.exp(vals[6]),
    }


def make_random_init(rng, true_log, noise):
    init = list(true_log)
    for i in [0, 1, 2, 3, 6]:
        init[i] = true_log[i] + rng.uniform(-noise, noise)
    for i in [4, 5]:
        scale = max(abs(true_log[i]), 0.05)
        init[i] = true_log[i] + rng.uniform(-2.0 * scale, 2.0 * scale)
    return init


def calculate_rmsre(params, truth, zero_thresh=0.01):
    est = backmap_params(params)
    keys = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
    e = np.array([est[k] for k in keys])
    t = np.array([truth[k] for k in keys])
    mask = np.abs(t) >= zero_thresh
    rmsre = float(np.sqrt(np.mean(((e[mask] - t[mask]) / np.abs(t[mask])) ** 2)))
    mae_zero = float(np.mean(np.abs(e[~mask] - t[~mask]))) if (~mask).any() else np.nan
    return rmsre, mae_zero, est


PARAM_KEYS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]


def make_summary_table(df, truth=true_dict, zero_thresh=0.01):
    rows = []
    if df.empty:
        return pd.DataFrame(rows)

    for model_name, g in df.groupby("model", sort=False):
        rms = g["rmsre"].dropna().astype(float)
        if len(rms):
            rows.append({
                "model": model_name,
                "param": "Overall",
                "n": int(len(g)),
                "true": np.nan,
                "mean": np.nan,
                "median": np.nan,
                "bias": np.nan,
                "RMSRE_mean": float(rms.mean()),
                "RMSRE_median": float(rms.median()),
                "P10": float(rms.quantile(0.10)),
                "P90": float(rms.quantile(0.90)),
                "P90_P10": float(rms.quantile(0.90) - rms.quantile(0.10)),
            })

        for key in PARAM_KEYS:
            col = f"{key}_est"
            if col not in g.columns:
                continue
            vals = g[col].dropna().astype(float)
            if vals.empty:
                continue
            tv = float(truth[key])
            mean_v = float(vals.mean())
            med_v = float(vals.median())
            p10 = float(vals.quantile(0.10))
            p90 = float(vals.quantile(0.90))

            if abs(tv) >= zero_thresh:
                rel_abs = np.abs((vals.to_numpy() - tv) / abs(tv))
                rmsre_mean = float(np.sqrt(np.mean(rel_abs ** 2)))
                rmsre_median = float(np.median(rel_abs))
            else:
                abs_err = np.abs(vals.to_numpy() - tv)
                rmsre_mean = float(np.mean(abs_err))
                rmsre_median = float(np.median(abs_err))

            rows.append({
                "model": model_name,
                "param": key,
                "n": int(len(vals)),
                "true": tv,
                "mean": mean_v,
                "median": med_v,
                "bias": mean_v - tv,
                "RMSRE_mean": rmsre_mean,
                "RMSRE_median": rmsre_median,
                "P10": p10,
                "P90": p90,
                "P90_P10": p90 - p10,
            })

    return pd.DataFrame(rows)

true_log = true_to_log_params(true_dict)
true_params = torch.tensor(true_log, dtype=DTYPE, device=DEVICE)
print(backmap_params(true_log))


{'sigmasq': 9.999999999999998, 'range_lat': 0.3, 'range_lon': 0.4, 'range_time': 2.0, 'advec_lat': 0.08, 'advec_lon': -0.126, 'nugget': 2.5}


In [18]:
def get_covariance_on_grid(lx, ly, lt, params):
    params = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat = lx - params[4] * lt
    u_lon = ly - params[5] * lt
    dist = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    return (phi1 / phi2) * torch.exp(-dist * phi2)


def build_target_grid(lat_range, lon_range):
    lats = torch.arange(min(lat_range), max(lat_range) + 1e-4, DELTA_LAT, device=DEVICE, dtype=DTYPE)
    lons = torch.arange(lon_range[0], lon_range[1] + 1e-4, DELTA_LON, device=DEVICE, dtype=DTYPE)
    lats = torch.round(lats * 10000) / 10000
    lons = torch.round(lons * 10000) / 10000
    lat_grid, lon_grid = torch.meshgrid(lats, lons, indexing="ij")
    grid_coords = torch.stack([lat_grid.flatten(), lon_grid.flatten()], dim=1)
    return lats, lons, grid_coords


def generate_field_values(n_lat, n_lon, t_steps, params):
    cpu = torch.device("cpu")
    f32 = torch.float32
    px, py, pt = 2 * n_lat, 2 * n_lon, 2 * t_steps

    lx = torch.arange(px, device=cpu, dtype=f32) * DELTA_LAT
    lx[px // 2:] -= px * DELTA_LAT
    ly = torch.arange(py, device=cpu, dtype=f32) * DELTA_LON
    ly[py // 2:] -= py * DELTA_LON
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt

    params_cpu = params.detach().cpu().float()
    Lx, Ly, Lt = torch.meshgrid(lx, ly, lt, indexing="ij")
    C = get_covariance_on_grid(Lx, Ly, Lt, params_cpu)
    S = torch.fft.fftn(C)
    S_real = torch.clamp(S.real, min=0.0)
    noise_fft = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(S_real) * noise_fft).real[:n_lat, :n_lon, :t_steps]
    return field.to(device=DEVICE, dtype=DTYPE)


def assemble_hourly_map(field, grid_coords, true_params):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid = grid_coords.shape[0]
    field_flat = field.reshape(n_grid, field.shape[-1])
    hourly = {}
    for t_idx in range(field.shape[-1]):
        rows = torch.zeros((n_grid, 4), device=DEVICE, dtype=DTYPE)
        rows[:, :2] = grid_coords
        rows[:, 2] = field_flat[:, t_idx] + torch.randn(n_grid, device=DEVICE, dtype=DTYPE) * nugget_std
        rows[:, 3] = float(t_idx)
        hourly[f"t{t_idx}"] = rows.detach().cpu()
    aggregated = torch.cat([hourly[k] for k in sorted(hourly)], dim=0)
    return [aggregated], [hourly]

lats_grid, lons_grid, grid_coords = build_target_grid(lat_range, lon_range)
n_lat, n_lon = len(lats_grid), len(lons_grid)
print(f"Target grid: {n_lat} x {n_lon} x {T_STEPS} = {n_lat * n_lon * T_STEPS:,} values")


Target grid: 114 x 159 x 8 = 145,008 values


In [19]:
def prepare_periodogram_2110(daily_agg, daily_hourly, params_seed):
    model = dw2110.debiased_whittle_likelihood()
    db = dw2110.debiased_whittle_preprocess(
        daily_agg, daily_hourly, day_idx=0, params_list=params_seed,
        lat_range=lat_range, lon_range=lon_range,
    )
    cur_df = db.generate_spatially_filtered_days(*lat_range, *lon_range).to(DEVICE)
    unique_times = torch.unique(cur_df[:, TIME_COL])
    time_slices = [cur_df[cur_df[:, TIME_COL] == t] for t in unique_times]

    J_vec, n1, n2, p_total, taper_grid, obs_masks = model.generate_Jvector_tapered_mv(
        time_slices, model.cgn_hamming, LAT_COL, LON_COL, VAL_COL, DEVICE,
    )
    I_sample = model.calculate_sample_periodogram_vectorized(J_vec)
    taper_auto = model.calculate_taper_autocorrelation_multivariate(taper_grid, obs_masks, n1, n2, DEVICE)
    return model, I_sample, taper_auto, n1, n2, p_total


def prepare_periodogram_grad(daily_agg, daily_hourly, params_seed):
    model = dwgrad.debiased_whittle_likelihood()
    db = dwgrad.debiased_whittle_preprocess(
        daily_agg, daily_hourly, day_idx=0, params_list=params_seed,
        lat_range=lat_range, lon_range=lon_range,
    )
    time_component_slices = [x.to(DEVICE) for x in db.generate_gradient_filtered_time_slices(*lat_range, *lon_range)]
    if not time_component_slices:
        raise ValueError("Gradient preprocessing returned no slices.")

    J_vec, n1, n2, p_total, taper_grid, obs_masks = model.generate_Jvector_tapered_mv(
        time_component_slices, model.cgn_hamming, LAT_COL, LON_COL, VAL_COL, DEVICE,
    )
    I_sample = model.calculate_sample_periodogram_vectorized(J_vec)
    taper_auto = model.calculate_taper_autocorrelation_multivariate(taper_grid, obs_masks, n1, n2, DEVICE)
    return model, I_sample, taper_auto, n1, n2, p_total


def prepare_periodogram_lat1(daily_agg, daily_hourly, params_seed):
    model = dwlat1.debiased_whittle_likelihood()
    db = dwlat1.debiased_whittle_preprocess(
        daily_agg, daily_hourly, day_idx=0, params_list=params_seed,
        lat_range=lat_range, lon_range=lon_range,
    )
    cur_df = db.generate_spatially_filtered_days(*lat_range, *lon_range).to(DEVICE)
    unique_times = torch.unique(cur_df[:, TIME_COL])
    time_slices = [cur_df[cur_df[:, TIME_COL] == t] for t in unique_times]

    J_vec, n1, n2, p_total, taper_grid, obs_masks = model.generate_Jvector_tapered_mv(
        time_slices, model.cgn_hamming, LAT_COL, LON_COL, VAL_COL, DEVICE,
    )
    I_sample = model.calculate_sample_periodogram_vectorized(J_vec)
    taper_auto = model.calculate_taper_autocorrelation_multivariate(taper_grid, obs_masks, n1, n2, DEVICE)
    return model, I_sample, taper_auto, n1, n2, p_total


def fit_dw(model_name, model, I_sample, taper_auto, n1, n2, p_total, initial_vals):
    params_list = [
        Parameter(torch.tensor([val], dtype=DTYPE, device=DEVICE), requires_grad=True)
        for val in initial_vals
    ]
    optimizer = torch.optim.LBFGS(
        params_list, lr=1.0, max_iter=lbfgs_eval, history_size=lbfgs_history,
        line_search_fn="strong_wolfe", tolerance_grad=1e-5,
    )
    start = time.time()
    nat_str, phi_str, raw_str, loss, steps = model.run_lbfgs_tapered(
        params_list=params_list, optimizer=optimizer,
        I_sample=I_sample, n1=n1, n2=n2, p_time=p_total,
        taper_autocorr_grid=taper_auto, max_steps=lbfgs_steps,
        device=DEVICE, grad_tol=1e-5,
    )
    elapsed = time.time() - start
    est_vec = torch.cat([p.detach().cpu() for p in params_list])
    rmsre, mae_zero, est = calculate_rmsre(est_vec, true_dict)
    return {
        "model": model_name,
        "loss": loss,
        "steps": steps,
        "time_s": elapsed,
        "rmsre": rmsre,
        "mae_zero_true": mae_zero,
        **{f"{k}_est": v for k, v in est.items()},
        "natural": nat_str,
        "raw": raw_str,
    }


## Run Controlled Iterations

All three models receive the same simulated field and the same initialization within each iteration.


In [20]:
records = []

out_dir = Path(config.mac_estimates_day_path)
out_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = out_dir / f"sim_dw_2110_grad_lat1_filter_{result_tag}_checkpoint.csv"

for it in range(num_iters):
    print("\n" + "=" * 70)
    print(f"Iteration {it + 1}/{num_iters}")
    print("=" * 70)

    initial_vals = make_random_init(rng, true_log, init_noise)
    init_nat = backmap_params(initial_vals)
    print(f"Init: sigmasq={init_nat['sigmasq']:.3f}, range_lon={init_nat['range_lon']:.3f}, "
          f"advec_lat={init_nat['advec_lat']:.3f}, advec_lon={init_nat['advec_lon']:.3f}, "
          f"nugget={init_nat['nugget']:.3f}")

    field = generate_field_values(n_lat, n_lon, T_STEPS, true_params)
    daily_agg, daily_hourly = assemble_hourly_map(field, grid_coords, true_params)
    del field

    print("\nPreparing DW_2110 periodogram...")
    model_2110, I_2110, taper_2110, n1_2110, n2_2110, p_2110 = prepare_periodogram_2110(
        daily_agg, daily_hourly, initial_vals,
    )
    print(f"DW_2110 grid={n1_2110}x{n2_2110}, p={p_2110}")

    print("Preparing DW_Gradient periodogram...")
    model_grad, I_grad, taper_grad, n1_grad, n2_grad, p_grad = prepare_periodogram_grad(
        daily_agg, daily_hourly, initial_vals,
    )
    print(f"DW_Gradient grid={n1_grad}x{n2_grad}, p={p_grad}")

    print("Preparing DW_Lat1 periodogram...")
    model_lat1, I_lat1, taper_lat1, n1_lat1, n2_lat1, p_lat1 = prepare_periodogram_lat1(
        daily_agg, daily_hourly, initial_vals,
    )
    print(f"DW_Lat1 grid={n1_lat1}x{n2_lat1}, p={p_lat1}")

    for model_name, model, I_s, taper_s, n1_s, n2_s, p_s in [
        ("DW_2110", model_2110, I_2110, taper_2110, n1_2110, n2_2110, p_2110),
        ("DW_Gradient", model_grad, I_grad, taper_grad, n1_grad, n2_grad, p_grad),
        ("DW_Lat1", model_lat1, I_lat1, taper_lat1, n1_lat1, n2_lat1, p_lat1),
    ]:
        print(f"\n--- Fit {model_name} ---")
        try:
            row = fit_dw(model_name, model, I_s, taper_s, n1_s, n2_s, p_s, initial_vals)
            row.update({
                "iter": it + 1,
                "n1": n1_s,
                "n2": n2_s,
                "p_total": p_s,
                "init_sigmasq": init_nat["sigmasq"],
                "init_range_lon": init_nat["range_lon"],
                "init_advec_lat": init_nat["advec_lat"],
                "init_advec_lon": init_nat["advec_lon"],
                "init_nugget": init_nat["nugget"],
            })
            records.append(row)
            print(f"{model_name}: loss={row['loss']} rmsre={row['rmsre']:.4f} time={row['time_s']:.1f}s")
        except Exception as exc:
            import traceback
            print(f"{model_name} failed: {type(exc).__name__}: {exc}")
            traceback.print_exc()

    df_live = pd.DataFrame(records)
    df_live.to_csv(checkpoint_file, index=False)
    display(df_live.tail(3)[["iter", "model", "loss", "rmsre", "advec_lat_est", "advec_lon_est", "time_s"]])

    running_summary = make_summary_table(df_live)
    display(running_summary)
    print(f"Checkpoint saved: {checkpoint_file}")



Iteration 1/30
Init: sigmasq=15.986, range_lon=0.436, advec_lat=0.232, advec_lon=0.006, nugget=1.416

Preparing DW_2110 periodogram...
DW_2110 grid=113x158, p=8
Preparing DW_Gradient periodogram...
DW_Gradient grid=113x158, p=16
Preparing DW_Lat1 periodogram...
DW_Lat1 grid=113x159, p=8

--- Fit DW_2110 ---
--- Step 1/8 ---
 Loss: 2.232451 | Max Grad: 1.033061e-02
  Params (Raw Log): log_phi1: 3.2139, log_phi2: 0.8777, log_phi3: 0.5941, log_phi4: -3.2262, advec_lat: 0.0867, advec_lon: -0.1226, log_nugget: 0.9045
--- Step 2/8 ---
 Loss: 1.942140 | Max Grad: 1.605671e-06
  Params (Raw Log): log_phi1: 3.2123, log_phi2: 0.8594, log_phi3: 0.6085, log_phi4: -3.2244, advec_lat: 0.0874, advec_lon: -0.1233, log_nugget: 0.9042
--- Step 3/8 ---
 Loss: 1.942094 | Max Grad: 1.605671e-06
  Params (Raw Log): log_phi1: 3.2123, log_phi2: 0.8594, log_phi3: 0.6085, log_phi4: -3.2244, advec_lat: 0.0874, advec_lon: -0.1233, log_nugget: 0.9042
--- Step 4/8 ---
 Loss: 1.942094 | Max Grad: 1.605671e-06
  Par

,iter,model,loss,rmsre,advec_lat_est,advec_lon_est,time_s
0,1,DW_2110,1.942,0.054404,0.087394,-0.123292,59.528904
1,1,DW_Gradient,-65.494,0.027708,0.084758,-0.122982,113.130477
2,1,DW_Lat1,-8.511,0.045929,0.087077,-0.120487,33.107445


,model,param,n,true,mean,median,bias,RMSRE_mean,RMSRE_median,P10,P90,P90_P10
0,DW_2110,Overall,1,NaN,NaN,NaN,NaN,0.054404,0.054404,0.054404,0.054404,0.0
1,DW_2110,sigmasq,1,10.000,10.516044,10.516044,0.516044,0.051604,0.051604,10.516044,10.516044,0.0
2,DW_2110,range_lat,1,0.300,0.312352,0.312352,0.012352,0.041173,0.041173,0.312352,0.312352,0.0
3,DW_2110,range_lon,1,0.400,0.423432,0.423432,0.023432,0.058581,0.058581,0.423432,0.423432,0.0
4,DW_2110,range_time,1,2.000,2.122988,2.122988,0.122988,0.061494,0.061494,2.122988,2.122988,0.0
5,DW_2110,advec_lat,1,0.080,0.087394,0.087394,0.007394,0.092421,0.092421,0.087394,0.087394,0.0
6,DW_2110,advec_lon,1,-0.126,-0.123292,-0.123292,0.002708,0.021489,0.021489,-0.123292,-0.123292,0.0
7,DW_2110,nugget,1,2.500,2.470034,2.470034,-0.029966,0.011986,0.011986,2.470034,2.470034,0.0
8,DW_Gradient,Overall,1,NaN,NaN,NaN,NaN,0.027708,0.027708,0.027708,0.027708,0.0
9,DW_Gradient,sigmasq,1,10.000,9.850145,9.850145,-0.149855,0.014985,0.014985,9.850145,9.850145,0.0


Checkpoint saved: /Users/joonwonlee/Documents/GEMS_TCO-1/outputs/day/estimates/save/day/sim_dw_2110_grad_lat1_filter_full_domain_advec_lon_neg0126_n60_3filters_checkpoint.csv

Iteration 2/30
Init: sigmasq=25.121, range_lon=0.673, advec_lat=0.126, advec_lon=0.037, nugget=4.544

Preparing DW_2110 periodogram...
DW_2110 grid=113x158, p=8
Preparing DW_Gradient periodogram...
DW_Gradient grid=113x158, p=16
Preparing DW_Lat1 periodogram...
DW_Lat1 grid=113x159, p=8

--- Fit DW_2110 ---
--- Step 1/8 ---
 Loss: 2.968003 | Max Grad: 9.511501e-03
  Params (Raw Log): log_phi1: 3.1385, log_phi2: 0.4958, log_phi3: 0.5908, log_phi4: -3.1795, advec_lat: 0.0762, advec_lon: -0.1213, log_nugget: 0.9478
--- Step 2/8 ---
 Loss: 1.968635 | Max Grad: 3.156808e-05
  Params (Raw Log): log_phi1: 3.1704, log_phi2: 0.7849, log_phi3: 0.5831, log_phi4: -3.2030, advec_lat: 0.0761, advec_lon: -0.1225, log_nugget: 0.9360
--- Step 3/8 ---
 Loss: 1.967811 | Max Grad: 3.156808e-05
  Params (Raw Log): log_phi1: 3.1704, l

,iter,model,loss,rmsre,advec_lat_est,advec_lon_est,time_s
3,2,DW_2110,1.968,0.097447,0.076111,-0.122469,48.474181
4,2,DW_Gradient,-65.609,0.017518,0.080600,-0.122428,93.680803
5,2,DW_Lat1,-8.479,0.087097,0.074903,-0.122635,28.500930


...

,model,param,n,true,mean,median,bias,RMSRE_mean,RMSRE_median,P10,P90,P90_P10
0,DW_2110,Overall,29,NaN,NaN,NaN,NaN,0.069245,0.059339,0.029514,0.133132,0.103618
1,DW_2110,sigmasq,29,10.000,10.162345,10.123532,0.162345,0.089178,0.054756,9.176648,11.570171,2.393523
2,DW_2110,range_lat,29,0.300,0.305485,0.307361,0.005485,0.097354,0.063427,0.272633,0.352122,0.079489
3,DW_2110,range_lon,29,0.400,0.406465,0.403884,0.006465,0.096850,0.073495,0.358228,0.458943,0.100714
4,DW_2110,range_time,29,2.000,2.042374,2.024905,0.042374,0.094630,0.064494,1.850001,2.330932,0.480931
5,DW_2110,advec_lat,29,0.080,0.079392,0.078470,-0.000608,0.068163,0.052555,0.072711,0.085714,0.013003
6,DW_2110,advec_lon,29,-0.126,-0.126305,-0.125700,-0.000305,0.053832,0.042880,-0.134516,-0.118203,0.016313
7,DW_2110,nugget,29,2.500,2.499696,2.494590,-0.000304,0.012897,0.009692,2.463987,2.547765,0.083778
8,DW_Gradient,Overall,29,NaN,NaN,NaN,NaN,0.055132,0.044115,0.024242,0.106605,0.082362
9,DW_Gradient,sigmasq,29,10.000,10.036893,9.855122,0.036893,0.076755,0.043611,9.318931,11.064561,1.745630


Checkpoint saved: /Users/joonwonlee/Documents/GEMS_TCO-1/outputs/day/estimates/save/day/sim_dw_2110_grad_lat1_filter_full_domain_advec_lon_neg0126_n60_3filters_checkpoint.csv

Iteration 30/30
Init: sigmasq=11.995, range_lon=0.790, advec_lat=-0.041, advec_lon=-0.123, nugget=3.206

Preparing DW_2110 periodogram...
DW_2110 grid=113x158, p=8
Preparing DW_Gradient periodogram...
DW_Gradient grid=113x158, p=16
Preparing DW_Lat1 periodogram...
DW_Lat1 grid=113x159, p=8

--- Fit DW_2110 ---
--- Step 1/8 ---
 Loss: 2.452867 | Max Grad: 5.888333e-03
  Params (Raw Log): log_phi1: 3.1807, log_phi2: 0.3239, log_phi3: 0.6159, log_phi4: -3.2975, advec_lat: 0.0783, advec_lon: -0.1249, log_nugget: 0.9531
--- Step 2/8 ---
 Loss: 2.127306 | Max Grad: 3.693078e-05
  Params (Raw Log): log_phi1: 3.2403, log_phi2: 0.9467, log_phi3: 0.6051, log_phi4: -3.3216, advec_lat: 0.0781, advec_lon: -0.1254, log_nugget: 0.9296
--- Step 3/8 ---
 Loss: 2.124201 | Max Grad: 3.693078e-05
  Params (Raw Log): log_phi1: 3.2403

,iter,model,loss,rmsre,advec_lat_est,advec_lon_est,time_s
87,30,DW_2110,2.124,0.024399,0.078077,-0.125360,52.488384
88,30,DW_Gradient,-65.233,0.047917,0.077963,-0.128484,92.830919
89,30,DW_Lat1,-8.326,0.027546,0.078310,-0.124143,28.409696


,model,param,n,true,mean,median,bias,RMSRE_mean,RMSRE_median,P10,P90,P90_P10
0,DW_2110,Overall,30,NaN,NaN,NaN,NaN,0.067750,0.056872,0.028126,0.132327,0.104201
1,DW_2110,sigmasq,30,10.000,10.153963,10.103815,0.153963,0.087694,0.053180,9.180741,11.545496,2.364755
2,DW_2110,range_lat,30,0.300,0.304859,0.305498,0.004859,0.096058,0.061999,0.272652,0.352027,0.079375
3,DW_2110,range_lon,30,0.400,0.405850,0.403375,0.005850,0.095379,0.070767,0.358232,0.457546,0.099313
4,DW_2110,range_time,30,2.000,2.042374,2.031629,0.042374,0.093120,0.062994,1.850677,2.324465,0.473787
5,DW_2110,advec_lat,30,0.080,0.079348,0.078451,-0.000652,0.067160,0.051349,0.072775,0.085504,0.012729
6,DW_2110,advec_lon,30,-0.126,-0.126274,-0.125621,-0.000274,0.052935,0.038530,-0.134146,-0.118234,0.015912
7,DW_2110,nugget,30,2.500,2.500821,2.494925,0.000821,0.012913,0.009796,2.464418,2.547501,0.083083
8,DW_Gradient,Overall,30,NaN,NaN,NaN,NaN,0.054891,0.045139,0.024607,0.104590,0.079983
9,DW_Gradient,sigmasq,30,10.000,10.053553,9.859690,0.053553,0.076099,0.043825,9.357578,11.040409,1.682831


Checkpoint saved: /Users/joonwonlee/Documents/GEMS_TCO-1/outputs/day/estimates/save/day/sim_dw_2110_grad_lat1_filter_full_domain_advec_lon_neg0126_n60_3filters_checkpoint.csv


In [21]:
df_results = pd.DataFrame(records)
if not df_results.empty:
    cols = [
        "iter", "model", "loss", "rmsre", "sigmasq_est", "range_lat_est",
        "range_lon_est", "range_time_est", "advec_lat_est", "advec_lon_est",
        "nugget_est", "time_s", "n1", "n2", "p_total",
    ]
    display(df_results[cols])

    summary = make_summary_table(df_results)
    display(summary)

    out_dir = Path(config.mac_estimates_day_path)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"sim_dw_2110_grad_lat1_filter_{result_tag}_results.csv"
    summary_file = out_dir / f"sim_dw_2110_grad_lat1_filter_{result_tag}_summary.csv"
    df_results.to_csv(out_file, index=False)
    summary.to_csv(summary_file, index=False)
    print(f"Saved: {out_file}")
    print(f"Saved: {summary_file}")
else:
    print("No completed fits.")


,iter,model,loss,rmsre,sigmasq_est,range_lat_est,range_lon_est,range_time_est,advec_lat_est,advec_lon_est,nugget_est,time_s,n1,n2,p_total
0,1,DW_2110,1.942,0.054404,10.516044,0.312352,0.423432,2.122988,0.087394,-0.123292,2.470034,59.528904,113,158,8
1,1,DW_Gradient,-65.494,0.027708,9.850145,0.293618,0.392981,1.972074,0.084758,-0.122982,2.477119,113.130477,113,158,16
2,1,DW_Lat1,-8.511,0.045929,10.186452,0.304528,0.419267,2.091460,0.087077,-0.120487,2.484305,33.107445,113,159,8
3,2,DW_2110,1.968,0.097447,10.864529,0.340792,0.456149,2.262732,0.076111,-0.122469,2.549879,48.474181,113,158,8
4,2,DW_Gradient,-65.609,0.017518,9.807025,0.298927,0.404218,2.055921,0.080600,-0.122428,2.492144,93.680803,113,158,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,29,DW_Gradient,-65.626,0.026157,10.137950,0.307604,0.417978,2.048676,0.081362,-0.129929,2.477379,86.745764,113,158,16
86,29,DW_Lat1,-8.484,0.059056,10.522113,0.321261,0.433144,2.157698,0.084754,-0.126712,2.510414,26.929734,113,159,8
87,30,DW_2110,2.124,0.024399,9.910866,0.286721,0.388020,2.042375,0.078077,-0.125360,2.533449,52.488384,113,158,8
88,30,DW_Gradient,-65.233,0.047917,10.536715,0.311355,0.422147,2.172102,0.077963,-0.128484,2.539665,92.830919,113,158,16


,model,param,n,true,mean,median,bias,RMSRE_mean,RMSRE_median,P10,P90,P90_P10
0,DW_2110,Overall,30,NaN,NaN,NaN,NaN,0.067750,0.056872,0.028126,0.132327,0.104201
1,DW_2110,sigmasq,30,10.000,10.153963,10.103815,0.153963,0.087694,0.053180,9.180741,11.545496,2.364755
2,DW_2110,range_lat,30,0.300,0.304859,0.305498,0.004859,0.096058,0.061999,0.272652,0.352027,0.079375
3,DW_2110,range_lon,30,0.400,0.405850,0.403375,0.005850,0.095379,0.070767,0.358232,0.457546,0.099313
4,DW_2110,range_time,30,2.000,2.042374,2.031629,0.042374,0.093120,0.062994,1.850677,2.324465,0.473787
5,DW_2110,advec_lat,30,0.080,0.079348,0.078451,-0.000652,0.067160,0.051349,0.072775,0.085504,0.012729
6,DW_2110,advec_lon,30,-0.126,-0.126274,-0.125621,-0.000274,0.052935,0.038530,-0.134146,-0.118234,0.015912
7,DW_2110,nugget,30,2.500,2.500821,2.494925,0.000821,0.012913,0.009796,2.464418,2.547501,0.083083
8,DW_Gradient,Overall,30,NaN,NaN,NaN,NaN,0.054891,0.045139,0.024607,0.104590,0.079983
9,DW_Gradient,sigmasq,30,10.000,10.053553,9.859690,0.053553,0.076099,0.043825,9.357578,11.040409,1.682831


Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/outputs/day/estimates/save/day/sim_dw_2110_grad_lat1_filter_full_domain_advec_lon_neg0126_n60_3filters_results.csv
Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/outputs/day/estimates/save/day/sim_dw_2110_grad_lat1_filter_full_domain_advec_lon_neg0126_n60_3filters_summary.csv
